# Day 2.3 — Embeddings and Semantic Search

Keyword search failed on "What keeps running during a blackout?" because the answer uses
different words. An embedding replaces word overlap with a numeric representation of
meaning, so paraphrases can land near each other.

```text
chunk text -> vector (once)      question -> vector (per query)
rank chunks by similarity between the two vectors
```

We index the corpus twice - with the offline hash embedder and with a trained
sentence-transformer model - and ask both the same question.

## Before you begin

### Learning outcomes

- Build a vector index and rank chunks by similarity instead of shared words.
- Compare `TokenHashEmbedder` with `SentenceTransformerEmbedder` on the same query and
  explain the difference in one sentence.

Architecture reference: [D06](../../diagrams/source/day_02.md).

### Expected observation

For the notebook 02 question, keyword search and the hash embedder both miss the correct
chunk; the trained model ranks it first. If the model cannot be downloaded, the notebook
prints why and continues with the hash embedder.

### Toggles

`EMBEDDER=auto|semantic|hash` chooses the embedder; `DAY2_MODE=mock` forces `hash`.
No API key is needed anywhere in this notebook.

## Concept briefing

## What embeddings do - and do not do

An embedding converts text into a vector so that a similarity function can rank nearby
representations. A trained semantic embedding may place paraphrases close together. The
course's deterministic token-hash embedder is different: it hashes each word into a fixed
position of a vector, so two texts are close only when they repeat the same words. It is
keyword matching in vector form, useful because it is stable offline, and it must not be
presented as a production semantic model.

Comparing the two embedders on the same query is the fastest way to see what a trained
model adds: the paraphrase that scores zero under word overlap can still rank first under
semantic similarity.

Similarity answers "which candidates are closest under this representation?" It does
not prove that a passage is relevant, sufficient or correct. Scores from different
models are not directly comparable, and there is no universal threshold.


In [ ]:
# --- Course setup: run this cell first -----------------------------------------
# 1) Locate this day's folder so we can import from src/ and read data/ no matter
#    where Jupyter, VS Code, or Colab started. Every file path below goes through
#    PROJECT_ROOT, never through the current working directory.
import os, sys
from pathlib import Path

def find_project_root(marker="src/knowledge_agent"):
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in [folder, *folder.glob("day_*")]:
            if (candidate / marker).exists():
                return candidate
    raise FileNotFoundError(
        "Course folder not found. On Google Colab run the 'Colab bootstrap' cell at the top "
        "of the day notebook first; locally, start Jupyter inside the repository folder."
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 2) Load the API key from the .env file at the repository root (Day 1.1 shows how to
#    create it). If no key is present we stay in deterministic MOCK mode: every cell
#    still runs, answers are fixed strings, and no credit is spent.
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
LIVE = bool(os.getenv("OPENROUTER_API_KEY"))

print("Project root :", PROJECT_ROOT)
print("Mode         :", "LIVE (OpenRouter)" if LIVE else "MOCK (no OPENROUTER_API_KEY found)")

# 3) Load the corpus and read the classroom toggles (EMBEDDER / DAY2_MODE).
from knowledge_agent.documents import load_markdown_corpus
from knowledge_agent.embeddings import TokenHashEmbedder, default_embedder_preference, load_embedder
from knowledge_agent.retrieval import VectorIndex, dot

chunks = load_markdown_corpus(PROJECT_ROOT / "data" / "corpus")
print("Chunks           :", len(chunks))
print("Embedder request :", default_embedder_preference(), "(set EMBEDDER=hash to force offline)")

## Step 1 — What a vector actually looks like

The hash embedder turns text into 512 numbers: each word is hashed to one position and
counted there, then the vector is scaled to length 1. Nothing was learned; it is a
bookkeeping trick that lets us do arithmetic on words.

In [ ]:
hash_embedder = TokenHashEmbedder(dimensions=512)
vector = hash_embedder.embed(["The controller stops charging at 45 degrees."])[0]

print("numbers per vector :", len(vector))
print("positions in use   :", sum(1 for value in vector if value != 0), "(one per distinct word)")
print("all other positions are exactly 0.0, so printing the first 12 shows nothing;")
print("here are the positions that the words actually landed on:")
for position, value in enumerate(vector):
    if value != 0:
        print(f"   position {position:3}  value {value:.3f}")

## Step 2 — Similarity is one multiplication away

Both vectors have length 1, so their dot product is the cosine similarity: 1.0 means
identical direction, 0.0 means nothing in common. Watch what the hash embedder thinks of
a paraphrase.

In [ ]:
def similarity(left, right):
    """Cosine similarity between two texts under the hash embedder."""
    left_vector, right_vector = hash_embedder.embed([left, right])
    return dot(left_vector, right_vector)

pairs = [
    ("Charging stops at 45 degrees.", "Charging stops at 45 degrees."),
    ("Charging stops at 45 degrees.", "The controller stops charging at 45 degrees."),
    ("What keeps running during a blackout?", "Priority 1 loads include emergency lighting."),
]
for left, right in pairs:
    print(f"{similarity(left, right):.3f}   {left!r}  vs  {right!r}")

print()
print("Identical text scores 1.0, shared words score in between, and the paraphrase")
print("scores 0.0 - the hash embedder is keyword matching in vector form.")

## Step 3 — Index the corpus with the hash embedder

`VectorIndex.add` embeds every chunk once and keeps the vectors in memory. `search`
embeds the question and sorts chunks by similarity. Ask the notebook 02 question again.

In [ ]:
PARAPHRASE = "What keeps running during a blackout?"          # same question as notebook 02
EXPECTED_CHUNK = "solar_microgrid:load-priorities"

hash_index = VectorIndex(hash_embedder)
hash_index.add(chunks)

def show(index, label, question, k=3):
    print(f"[{label}] {question}")
    for item in index.search(question, top_k=k):
        print(f"   rank {item.rank}  score {item.score:.3f}  {item.chunk.chunk_id:42} {item.chunk.section}")

def rank_of(index, question, chunk_id):
    """Position of one chunk in the full ranking (1 = best)."""
    full = index.search(question, top_k=len(chunks))
    return next(item.rank for item in full if item.chunk.chunk_id == chunk_id)

show(hash_index, "hash", PARAPHRASE)
print()
print("Rank of", EXPECTED_CHUNK, "->", rank_of(hash_index, PARAPHRASE, EXPECTED_CHUNK), "of", len(chunks))

## Step 4 — Load a trained embedder

`load_embedder` asks for a real sentence-transformer model. The first call downloads about
90 MB; afterwards it is cached. If the download fails (no network, no package) it returns
the hash embedder instead and prints why, so the rest of the notebook still runs.

In [ ]:
embedder, embedder_label = load_embedder()          # honours EMBEDDER / DAY2_MODE
semantic_available = embedder_label == "semantic"

semantic_index = None
if semantic_available:
    semantic_index = VectorIndex(embedder)
    semantic_index.add(chunks)
    print("Semantic index built with", len(semantic_index.chunks), "chunks.")
else:
    print("No semantic index: the comparison below repeats the hash results and says so.")

## Step 5 — The same question through three retrievers

Keyword overlap, hash vectors, semantic vectors. Only the last one connects *blackout*
with *islanded operation* and *priority 1 loads*, because it was trained on text where
those ideas co-occur.

In [ ]:
from knowledge_agent.text import content_terms

# Rebuild the notebook 02 lexical ranking so all three sit in one table.
def keyword_rank(question, chunk_id):
    scored = sorted(
        ((len(content_terms(question) & content_terms(chunk.searchable_text)), chunk) for chunk in chunks),
        key=lambda pair: pair[0],
        reverse=True,
    )
    return [chunk.chunk_id for _, chunk in scored].index(chunk_id) + 1

print("Question:", PARAPHRASE)
print()
print(f"{'retriever':22}{'rank of expected chunk':26}top-1 chunk")
print("-" * 90)
print(f"{'keyword overlap':22}{keyword_rank(PARAPHRASE, EXPECTED_CHUNK):<26}"
      f"{max(chunks, key=lambda c: len(content_terms(PARAPHRASE) & content_terms(c.searchable_text))).chunk_id}")
print(f"{'hash embedder':22}{rank_of(hash_index, PARAPHRASE, EXPECTED_CHUNK):<26}"
      f"{hash_index.search(PARAPHRASE, top_k=1)[0].chunk.chunk_id}")
if semantic_index is not None:
    print(f"{'semantic embedder':22}{rank_of(semantic_index, PARAPHRASE, EXPECTED_CHUNK):<26}"
          f"{semantic_index.search(PARAPHRASE, top_k=1)[0].chunk.chunk_id}")
    print()
    show(semantic_index, "semantic", PARAPHRASE)
else:
    print(f"{'semantic embedder':22}{'not available':26}-")

## Step 6 — A score is a ranking, not a verdict

Similarity says "closest under this representation". It does not say "this passage
answers the question". Ask something the corpus cannot answer and watch the top score
stay comfortably positive.

In [ ]:
indexes = [("hash", hash_index)]
if semantic_index is not None:
    indexes.append(("semantic", semantic_index))

questions = [
    "How long are battery fault records retained?",     # answerable
    "What is the purchase price of the battery?",       # not in the corpus at all
]
for label, current in indexes:
    for question in questions:
        best = current.search(question, top_k=1)[0]
        print(f"[{label:9}] best score {best.score:.3f} -> {best.chunk.chunk_id:42} {question}")
    print()

print("Two things to notice:")
print(" 1. the unanswerable question never scores 0 - the closest chunk is always returned;")
print(" 2. the same pair of questions produces different numbers under each embedder, so a")
print("    cut-off tuned for one is meaningless for the other.")
print("Notebook 05 lets the generator judge the evidence, and notebook 06 measures retrieval")
print("against known answers instead of trusting a decimal.")

## Step 7 — Optional: the same vectors inside Chroma

Our in-memory index keeps the mathematics visible. A vector database stores the same
vectors and adds persistence, filtering and scale. It is optional for this course.

In [ ]:
try:
    from knowledge_agent.retrieval import ChromaVectorIndex
    chroma_index = ChromaVectorIndex(embedder, collection_name="day2_lab")
    chroma_index.add(chunks)
    for item in chroma_index.search(PARAPHRASE, top_k=3):
        print(f"   rank {item.rank}  score {item.score:.3f}  {item.chunk.chunk_id}")
    print("Same embedder, same chunks - only the storage layer changed.")
except Exception as exc:
    print("Chroma not available:", type(exc).__name__)
    print("Optional: pip install chromadb to run this part. Nothing else in Day 2 needs it.")

### Try it yourself

The Errors section says a rejected command must not simply be repeated. Ask about that in
everyday words - no "error", no "retry", no "command" - and predict which retriever finds it.

In [ ]:
# --- Worked solution ---
MY_QUESTION = "What happens if someone presses the wrong button twice?"
TARGET = "controller_interface:errors"

print("Question:", MY_QUESTION)
print("hash rank    :", rank_of(hash_index, MY_QUESTION, TARGET), "of", len(chunks))
if semantic_index is not None:
    print("semantic rank:", rank_of(semantic_index, MY_QUESTION, TARGET), "of", len(chunks))
    show(semantic_index, "semantic", MY_QUESTION)
else:
    print("semantic rank: not available in this environment")
    show(hash_index, "hash", MY_QUESTION)

print()
print("The section never says 'button' or 'twice'; it says 'Repeating a rejected operational")
print("command ... is prohibited'. Word overlap cannot bridge that, so the hash embedder")
print("buries the chunk far down the list while the trained model puts it near the top.")

### Checkpoint

**1. Which component creates vectors, and which one stores and searches them?**

<details><summary>Show answer</summary>

The *embedder* creates vectors (`TokenHashEmbedder`, `SentenceTransformerEmbedder`); the
*index* stores them and ranks by similarity (`VectorIndex`, `ChromaVectorIndex`). They are
separate on purpose: you can swap Chroma in without touching the embedder, and swap the
embedder without touching the storage - which is exactly the experiment in Step 5.

</details>

**2. Step 6 showed the unanswerable question still scoring well above zero. Could we just
reject everything below a fixed cut-off such as 0.5?**

<details><summary>Show answer</summary>

No. Scores depend on the model, the text length and the query, and they are not calibrated
probabilities. A threshold tuned on three questions will break on the fourth. Decide with
evidence instead: check the retrieved text (notebook 05) and measure against a golden set
(notebook 06).

</details>

### Recap

- **Limitation we saw:** word overlap - including the hash embedder, which is word overlap
  in disguise - cannot connect "blackout" with "islanded operation".
- **Layer we added:** a trained embedding model behind the same `VectorIndex` interface.
- **Evidence it worked:** the expected chunk moves from rank 14 (keyword) to rank 1 under
  semantic similarity for the identical question.